# 03 · Similarity by location

**Purpose.** Decide how to score the **location** block: how well a marketed listing's location fits the
portfolio, for the brief's prompt 2 (a homogeneity score for each dimension, location among them) and as one
input to prompt 1 (the most similar properties).

The data is the cleaned output of `indurent_matching.data` (see `01_eda.ipynb` for the cleaning decisions,
section 2.2 for the location checks). The notebook ends with decisions to carry into
`src/indurent_matching/features.py`.

## 1. The question and the usable data

**Location homogeneity** asks: *does Indurent own stock near this listing?* A listing in an area where the
portfolio is dense is homogeneous with it on location; one far from any portfolio unit is not.

Only locations we trust are used: **high** confidence (inside the labelled region) and **medium** (on land, but
outside the labelled region). **Unknown** locations (no coordinates, or 1 km+ offshore) have no location score.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from indurent_matching.data import load_marketed, load_portfolio

# Store each chart's data in the notebook itself; JupyterLab draws it with the Plotly extension, and the HTML
# copy is a fallback for other viewers.
pio.renderers.default = "plotly_mimetype+notebook_connected"

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
portfolio = load_portfolio(ROOT / "data" / "Current Portfolio.xlsx")
marketed = load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx")

located = portfolio[portfolio["location_confidence"] != "unknown"]
print(f"Portfolio: {len(located):,} of {len(portfolio):,} units have a usable location, "
      f"in {located['industrial_estate_name'].nunique()} of {portfolio['industrial_estate_name'].nunique()} estates")
portfolio["location_confidence"].value_counts().to_frame("units")

Portfolio: 892 of 1,250 units have a usable location, in 151 of 263 estates


,units
location_confidence,
medium,652
unknown,358
high,240


In [2]:
marketed[["industrial_estate_name", "region", "location_confidence", "derived_region", "nearest_city",
          "miles_to_city"]].round(1)

,industrial_estate_name,region,location_confidence,derived_region,nearest_city,miles_to_city
listing_id,,,,,,
M1,Cherry Lane,North West,medium,Wales,Cardiff,72.8
M2,Tech Hub,South East,unknown,None,NaN,NaN
M3,Spitfire Park,Midlands,high,Midlands,Manchester,14.0
M4,Stable Lane,South East,medium,East of England,London,40.6
M5,Chancery Depot,NaN,unknown,None,NaN,NaN


* **892 of 1,250 portfolio units** (151 of 263 estates) have a usable location: 240 high confidence, 652 medium.
  The other 358 can't take part in location comparisons, either as matches or in the counts a listing is
  scored against.
* Three listings have a usable location:
  * **M1 (Cherry Lane)**: medium; labelled North West but its coordinates fall in **Wales**.
  * **M3 (Spitfire Park)**: high; Midlands, 14 miles from Manchester.
  * **M4 (Stable Lane)**: medium; labelled South East but its coordinates fall in the **East of England**.
* **M2 (Tech Hub) has no coordinates and M5 (Chancery Depot) is 27 km offshore**, so their location score is
  "not available", not a low score.

## 2. Where the portfolio is

In [3]:
fig = px.scatter_geo(located, lat="latitude", lon="longitude", color="location_confidence",
                     hover_name="industrial_estate_name", hover_data=["unit_name", "derived_region"],
                     height=750, title="Portfolio units with a usable location, and the listings (stars)")
fig.update_traces(marker={"size": 5, "opacity": 0.6})
listings = marketed[marketed["location_confidence"] != "unknown"]
fig.add_scattergeo(lat=listings["latitude"], lon=listings["longitude"], name="marketed listing",
                   text=listings["listing_id"] + " " + listings["industrial_estate_name"], hoverinfo="text",
                   marker={"symbol": "star", "size": 16, "color": "firebrick", "line": {"width": 1, "color": "white"}})
fig.update_geos(resolution=50, showland=True, landcolor="#f2f2f2", showcountries=True,
                lataxis_range=[49.8, 59], lonaxis_range=[-7, 2.2])
fig.show()

In [4]:
by_region = located.groupby("derived_region").agg(units=("property_id", "size"),
                                                   estates=("industrial_estate_name", "nunique"))
by_region["% of units"] = (by_region["units"] / by_region["units"].sum() * 100).round(0)
by_region.sort_values("units", ascending=False)

,units,estates,% of units
derived_region,,,
Midlands,279,41,31.0
East of England,175,28,20.0
South West,107,20,12.0
South East,96,18,11.0
North West,84,11,9.0
Wales,82,7,9.0
North East,26,8,3.0
Yorks. & Humber,24,11,3.0
London,11,3,1.0


* The portfolio is **concentrated in the Midlands (31% of located units) and the East of England (20%)**, with
  the South West, South East, North West and Wales at 9-12% each. The North East, Yorkshire, London and
  Scotland together hold under 10%.
* By region, the listings sit in quite different places:
  * **M3 (Midlands)** is in the portfolio's heartland;
  * **M4 (East of England)** is in its second-biggest region;
  * **M1 (Wales)** is in a region with 82 units but only 7 estates.
* Regions are coarse, though: a listing can be in a busy region but far from any unit in it. The score should
  use actual distances (section 3), not region membership.

In [5]:
estate_sizes = located["industrial_estate_name"].value_counts()
print(f"Largest estates: {', '.join(f'{name} ({n})' for name, n in estate_sizes.head(5).items())}")
print(f"The 5 largest estates hold {estate_sizes.head(5).sum() / len(located):.0%} of located units; "
      f"{(estate_sizes == 1).sum()} estates are a single unit")

Largest estates: Quartz Wharf 16 (64), Ruby Arc 96 (48), Velvet Park 60 (41), Granite Pointe 39 (37), Crimson Forge 9 (37)
The 5 largest estates hold 25% of located units; 57 estates are a single unit


* **A few large estates hold a big share of the units**: the 5 largest hold a quarter of all located units
  (Quartz Wharf 16 alone has 64), while 57 estates are a single unit.
* That matters for any score built by counting nearby units: a listing next to Quartz Wharf 16 would look very
  homogeneous on location because of one estate, much as the 2022 spike inflated one build year. Counting
  **estates** instead of units, or both, is worth comparing when we build the score (section 4).

## 3. Each listing sits on an existing estate

Before scoring, a check: how close is each listing to the nearest portfolio unit, and which estate is it in?
This uses every listing with coordinates, including M5 (offshore), since the question is only which portfolio
units share its coordinates.

In [6]:
from indurent_matching.geo import haversine_miles

with_coords = portfolio.dropna(subset=["latitude", "longitude"])
rows = []
for listing_id, listing in marketed.dropna(subset=["latitude", "longitude"]).iterrows():
    miles = haversine_miles(listing["latitude"], listing["longitude"], with_coords["latitude"], with_coords["longitude"])
    host = with_coords.loc[miles.idxmin(), "industrial_estate_name"]
    host_units = portfolio[portfolio["industrial_estate_name"] == host]
    rows.append({
        "listing": f"{listing_id} {listing['industrial_estate_name']}",
        "listing unit": listing["unit_name"],
        "metres to nearest portfolio unit": round(miles.min() * 1609),
        "sits on estate": host,
        "units in estate": len(host_units),
        "listing region label": listing["region"],
        "estate region label": host_units["region"].iloc[0],
        "location confidence": listing["location_confidence"],
    })
pd.DataFrame(rows).set_index("listing")

,listing unit,metres to nearest portfolio unit,sits on estate,units in estate,listing region label,estate region label,location confidence
listing,,,,,,,
M1 Cherry Lane,Unit 10,40,Scarlet Pavilion 78,9,North West,North West,medium
M3 Spitfire Park,Unit 42,9,Velvet Park 60,41,Midlands,Midlands,high
M4 Stable Lane,Unit 14,21,Sable Mill 91,10,South East,South East,medium
M5 Chancery Depot,Unit 2,71,Granite Depot 69,1,NaN,Wales,unknown


* **Every listing with coordinates sits on an existing Indurent estate**: its nearest portfolio unit is within
  about 50 m, the same spread as units within one estate.
* The **unit numbers follow on from the estate's own**: Velvet Park 60 has Units 1-41 and M3 is Unit 42;
  Scarlet Pavilion 78 has 1-9 and M1 is Unit 10; Granite Depot 69 has Unit 1 and M5 is Unit 2. (M4 is Unit 14
  on an estate of 10.)
* The **region labels match the estate's too**, including the wrong ones: M1 and Scarlet Pavilion 78 are both
  labelled North West but sit in Wales.
* **M5's estate, Granite Depot 69, has the same offshore coordinates**, so M5 stays "not available" for location.
* So these listings look like **further units on estates Indurent already owns** (infill acquisitions). That's
  the single most useful location fact for an answer: *"M3 sits on Velvet Park 60, an estate you already own."*
* It also means **nearest portfolio units are the wrong neighbours**: they would all be from the listing's own
  estate, at about 0 miles, so every listing would get the same perfect score. Section 4 uses **estates** as the
  neighbours instead, and leaves out the estate the listing sits on.

## 4. Distance to the nearest estates

Each estate becomes one point (the average of its units' coordinates, usable locations only). For a listing,
we measure the distance to its **k nearest estates, leaving out the one it sits on**; for comparison, each
portfolio estate is measured against its k nearest *other* estates.

In [7]:
estates = located.groupby("industrial_estate_name")[["latitude", "longitude"]].mean()


def nearest_estates(lat, lon, exclude, k):
    """Distances in miles to the k nearest estates, leaving out ``exclude``."""
    others = estates.drop(exclude)
    return np.sort(haversine_miles(lat, lon, others["latitude"], others["longitude"]))[:k]


hosts = pd.DataFrame(rows).assign(listing_id=lambda t: t["listing"].str.split().str[0]).set_index("listing_id")
usable = marketed[marketed["location_confidence"] != "unknown"]
table = pd.DataFrame({
    f"mean miles, {k} nearest": [
        nearest_estates(r["latitude"], r["longitude"], hosts.loc[i, "sits on estate"], k).mean()
        for i, r in usable.iterrows()
    ]
    for k in [1, 5, 10]
}, index=usable.index).round(1)
table.insert(0, "sits on estate", hosts.loc[usable.index, "sits on estate"])
table

,sits on estate,"mean miles, 1 nearest","mean miles, 5 nearest","mean miles, 10 nearest"
listing_id,,,,
M1,Scarlet Pavilion 78,4.4,36.2,48.2
M3,Velvet Park 60,9.1,13.6,20.3
M4,Sable Mill 91,6.3,11.5,17.3


* **k matters, mostly for M1.** Its single nearest other estate is only 4.4 miles away (the closest of the
  three), but its 5 nearest average 36 miles: one neighbour, then a big gap. With k = 1, M1 would look like
  the best-placed listing; with k = 5 or 10, the most isolated.
* **k = 1 is fragile**: one estate decides the score. **k = 5 or 10** describe the surrounding area, which is
  what "homogeneous on location" should mean. M3 and M4 rank the same either way.

In [8]:
K = 5  # decided: mean of the 5 nearest other estates
portfolio_mean = pd.Series(
    [nearest_estates(r["latitude"], r["longitude"], name, K).mean() for name, r in estates.iterrows()],
    index=estates.index,
)
fig = px.histogram(portfolio_mean, nbins=40, height=380,
                   title=f"Portfolio estates: mean miles to their {K} nearest other estates (dashed lines: listings)",
                   labels={"value": "Mean miles"})
fig.update_layout(showlegend=False, yaxis_title="Estates")
for miles in table[f"mean miles, {K} nearest"]:
    fig.add_vline(x=miles, line_dash="dash", line_color="firebrick")
fig.show()
print(f"Portfolio estates: median {portfolio_mean.median():.1f} miles, "
      f"10th-90th percentile {portfolio_mean.quantile(0.1):.1f}-{portfolio_mean.quantile(0.9):.1f}, "
      f"max {portfolio_mean.max():.1f} ({portfolio_mean.idxmax()})")

Portfolio estates: median 16.1 miles, 10th-90th percentile 11.1-31.9, max 126.8 (Sable Forge 35)


* A typical portfolio estate has its 5 nearest other estates **about 16 miles away on average**; 80% fall
  between 11 and 32 miles. A long tail of isolated estates runs out to 127 miles (Sable Forge 35).
* **M4 (11.5 miles) and M3 (13.6 miles)** are in typical-to-dense parts of the portfolio, besides sitting on an
  estate Indurent owns.
* **M1 (36.2 miles)** is beyond 90% of portfolio estates: apart from Scarlet Pavilion 78, the estate it sits on,
  and one other estate 4 miles away, Indurent owns little nearby.

## 5. The location score

**Decision:** the score uses the **mean distance to the 5 nearest other estates**, rescaled so that the
best-placed estate in the portfolio scores 1:

score = (best portfolio estate's mean distance) / (listing's mean distance)

The score halves each time the distance doubles: 0.5 means the listing's nearest estates are on average twice as
far away as for the best-placed estate. This matches the age score, where the busiest band scores 1. The nearest
other estate is reported alongside as context, not scored.

In [9]:
best = portfolio_mean.min()
portfolio_score = best / portfolio_mean
print(f"Best-placed estate: {portfolio_mean.idxmin()} ({best:.1f} miles to its 5 nearest on average). "
      f"Portfolio scores: median {portfolio_score.median():.2f}, "
      f"10th-90th percentile {portfolio_score.quantile(0.1):.2f}-{portfolio_score.quantile(0.9):.2f}")

scores = table[["sits on estate", "mean miles, 1 nearest", f"mean miles, {K} nearest"]].rename(
    columns={"mean miles, 1 nearest": "nearest other estate (miles)"})
scores["location score"] = (best / scores[f"mean miles, {K} nearest"]).round(2)
# Context, not scored: the share of portfolio estates the listing is better placed than.
scores["better placed than (% of estates)"] = [
    round((portfolio_score < s).mean() * 100) for s in best / scores[f"mean miles, {K} nearest"]
]
scores

Best-placed estate: Slate Hub 99 (8.5 miles to its 5 nearest on average). Portfolio scores: median 0.52, 10th-90th percentile 0.27-0.76


,sits on estate,nearest other estate (miles),"mean miles, 5 nearest",location score,better placed than (% of estates)
listing_id,,,,,
M1,Scarlet Pavilion 78,4.4,36.2,0.23,7
M3,Velvet Park 60,9.1,13.6,0.62,63
M4,Sable Mill 91,6.3,11.5,0.73,85


In [10]:
fig = px.histogram(portfolio_score, nbins=30, height=380,
                   title="Location score of portfolio estates (dashed lines: listings)",
                   labels={"value": "Location score"})
fig.update_layout(showlegend=False, yaxis_title="Estates")
for score in scores["location score"]:
    fig.add_vline(x=score, line_dash="dash", line_color="firebrick")
fig.show()

* The best-placed estate, **Slate Hub 99**, has its 5 nearest estates 8.5 miles away on average and scores 1.
  A typical portfolio estate scores **about 0.5**; 80% score between 0.27 and 0.76.
* The listings: **M4 0.73** and **M3 0.62**, both better placed than a typical portfolio estate; **M1 0.23**,
  in the bottom 10%.
* The **percentile** (share of portfolio estates the listing is better placed than) is context for an answer,
  not the score: it stretches small differences (M3 and M4 are 2 miles apart but would score 63% vs 85%).
* In an answer, the score comes with its context, e.g. *"M1 sits on Scarlet Pavilion 78, which you own. The
  next nearest estate is 4.4 miles away, but beyond that Indurent owns little nearby: location score 0.23,
  better placed than only 7% of Indurent's estates."*
* **M2 and M5: not available** (no coordinates; offshore).

## 6. Decisions for `features.py`

| Topic | Decision | Why |
|---|---|---|
| Usable locations | High and medium confidence only (892 units, 151 estates) | Unknown locations (no coordinates, or 1 km+ offshore) can't be placed (EDA 2.2) |
| Neighbours | Estates, not units; each estate is the average of its units' coordinates | Every listing sits on an existing estate, so the nearest units are all from that estate at about 0 miles, and every listing would score the same |
| The estate a listing sits on | Reported as a fact ("sits on Velvet Park 60, which you own"), left out of the score | The most useful location fact for an answer; leaving it out makes listings comparable with portfolio estates, which are measured against other estates |
| Distance | Mean miles to the 5 nearest other estates | Describes the surrounding area; a single nearest estate can mislead (M1: 4.4 miles to one, then 30+ miles to the rest) |
| Score | Best portfolio estate's mean distance / listing's mean distance (best = 1) | Same idea as the age score; halves each time the distance doubles; not set by the extreme isolated estates |
| Context, not scored | The nearest other estate (miles); the percentile (better placed than X% of portfolio estates) | Readable facts for answers; the percentile isn't the score because it stretches small differences |
| No usable location | Location score "not available" (M2, M5), never 0 | A missing location says nothing about where the property is |